In [0]:
#Notebook : raw_to_landing_cross
# Task: Ingest 6 cross domain files from raw to landing
# Author: Virendra Dilip Tambavekar
# HRM ID: 6217
# Date : 15/05/2026

In [0]:
#Importing required libraries
import logging
import uuid
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import dataframe
#Initialize Logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [0]:
dbutils.widgets.text("catalog","charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id","1")

catalog = dbutils.widgets.get("catalog")
batch_id = dbutils.widgets.get("batch_id")

#Generating unique run id
run_id = str(uuid.uuid4())

#Paths
source_base_path = f"abfss://raw@schwabdldevsa.dfs.core.windows.net/Batch{batch_id}/"
target_base_path = f"/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/Batch{batch_id}"

In [0]:
#Function to generate string schemas to prevent schema inference data loss
def get_string_schema(columns: list) -> StructType :
    return StructType([StructField(col, StringType(), True) for col in columns])

#Domain Mapping
cross_domain_files = {
    "Date.txt" : {
        "has_header" : True,
        "schema" : None
    },
    "Time.txt" : {
        "has_header" : True,
        "schema" : None
    },
    "StatusType.txt" : {
        "has_header" : False,
        "schema" : get_string_schema(["ST_ID", "ST_NAME"])
    },
    "TaxRate.txt" : {
         "has_header" : False,
        "schema" : get_string_schema(["TX_ID", "TX_NAME", "TX_RATE"])
    },
    "Industry.txt" : {
        "has_header" : False,
        "schema" : get_string_schema(["IN_ID","IN_NAME","IN_SC_ID"])
    },
    "TradeType.txt" : {
        "has_header" : False,
        "schema" : get_string_schema(["TT_ID","TT_NAME","TT_IS_SELL","TT_IS_MRKT"])
    }
}

In [0]:
def process_landing_file(file_name :str, config:dict) -> dict:
    source_path = f"{source_base_path}{file_name}"
    target_path = f"{target_base_path}/{file_name.split('.')[0]}"
    #Exception Handling for files
    try:
        dbutils.fs.ls(source_path)
    except Exception:
        logger.info(f"{file_name} not found in Batch{batch_id}")
        return {"file": file_name, "status" : "SKIPPED"}
    logger.info(f"Processing {file_name}")
    #Read raw file
    if config["has_header"]:
        raw_df = spark.read.csv(source_path, sep="|", header = True)
    else :
        raw_df = spark.read.csv(source_path, sep="|", header= False, schema=config["schema"])

    #Metadata adding
    landing_df = (
        raw_df
        .withColumn("_landing_ts", current_timestamp())
        .withColumn("_batch",lit(batch_id))
        .withColumn("_source_file",lit(file_name))
        .withColumn("_run_id",lit(run_id))
    )
    landing_df.write.mode("overwrite").parquet(target_path)

    #Validation Read
    record_count = spark.read.parquet(target_path).count()
    #display(record_count)
    logger.info(f"Successfully landed {file_name} with {record_count} records")

    return {"file" : file_name, "status": "SUCCESS","count" : record_count}

In [0]:
def main():
    logger.info(f"Cross Domain Raw to Landing ingestion | Run id : {run_id}")
    results = []
    for file_name, config in cross_domain_files.items():
        try:
            result = process_landing_file(file_name,config)
            results.append(result)
        except Exception as e:
            logger.error(f"Failed to process {file_name} : {str(e)}")
            raise e
    #Execution Summary Display
    if results :
        summary_df = spark.createDataFrame(results)
        display(summary_df)

        success_count = len([r for r in results if r["status"] == "SUCCESS"])
        if batch_id == "1" and success_count != 6:
            logger.warning("File missing for Cross Domain in Batch 1")
    logger.info("Cross Domain Landing pipeline Completed")

main()
